## Start loading the data and to merge with only the necessary columns

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
# This script loads patient data from parquet files in a specified folder
def load_all_patient_data(folder_path):
    """
    Load all patient data from parquet files in the specified folder.

    Parameters:
    folder_path (str): Path to the folder containing parquet files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all files.
    """
    all_data = []
    try:
        for filename in os.listdir(folder_path):
            if filename.endswith('.parquet'):
                file_path = os.path.join(folder_path, filename)
                df = pd.read_parquet(file_path)
                all_data.append(df)
        
        if all_data:
            combined_data = pd.concat(all_data, ignore_index=True)
            return combined_data
        else:
            print("No .parquet files found in the folder.")
            return None
    except Exception as e:
        print(f"Error loading data: {e}")
        return None

if __name__ == "__main__":
    folder_path = './Readmitted_patients'
    data = load_all_patient_data(folder_path)

    if data is None:
        print("Failed to load data.")
    else:
        print("Data loaded successfully.")

        # Convert charttime to datetime and valuenum to numeric
        data['charttime'] = pd.to_datetime(data['charttime'], errors='coerce')
        data['valuenum'] = pd.to_numeric(data['valuenum'], errors='coerce')

        # Calculate the timeline as the difference in hours from max(charttime)
        data['timeline'] = data.groupby('SUBJECT_ID')['charttime'].transform(lambda x: (x.max() - x).dt.total_seconds() // 3600)
        data ['Label'] = 1
        d_items_path = './full_tables/d_items.csv'
        # preserve original itemid and replace itemid values with the human-readable label from items_df (if available)
        if 'itemid' in data.columns:
            # ensure items_df is available (use existing items_df if present, otherwise try to load d_items_path)
            if 'items_df' not in globals():
                if os.path.exists(d_items_path):
                    items_df = pd.read_csv(d_items_path, dtype={'itemid': str})
                else:
                    items_df = pd.DataFrame(columns=['itemid', 'label'])

            # build mapping of itemid -> label (both as strings)
            if 'itemid' in items_df.columns and 'label' in items_df.columns:
                label_map = items_df.set_index(items_df['itemid'].astype(str))['label'].to_dict()
            else:
                label_map = {}

            data['itemid'] = data['itemid'].astype(str).map(label_map)
            data.rename(columns={'itemid': 'Item_label'}, inplace=True)
            data = data[['SUBJECT_ID','timeline','Item_label','valuenum', 'Label']]
        else:
            print("Warning: 'itemid' column not found in data; skipping label mapping.")
            items_df = pd.DataFrame(columns=['itemid', 'label'])

        print("Data after processing charttime and itemid:")
        print(data.head())
        
        
    
        



Data loaded successfully.
Data after processing charttime and itemid:
  SUBJECT_ID  timeline              Item_label  valuenum  Label
0   10022620      16.0  Temperature Fahrenheit      97.5      1
1   10022620      15.0              Heart Rate      87.0      1
2   10022620      15.0        Respiratory Rate      21.0      1
3   10022620      15.0              Heart Rate      94.0      1
4   10022620      15.0        Respiratory Rate      27.0      1
